In [1]:
from pathlib import Path

ROOT = Path.cwd()
DATA_DIR = ROOT / "parquet"

In [2]:
import pandas as pd

amazon_books = pd.read_parquet(DATA_DIR / "amazon.parquet")
goodreads = pd.read_parquet(DATA_DIR / "goodreads.parquet")
metabooks = pd.read_parquet(DATA_DIR / "metabooks.parquet")

In [3]:
metabooks.columns

Index(['id', 'title', 'author_name', 'average_rating', 'language', 'genres',
       'publisher', 'page_count', 'price', 'publish_year', 'isbn_clean'],
      dtype='object')

In [4]:
amazon_books.columns

Index(['id', 'title', 'book-author', 'year-of-publication', 'publisher',
       'isbn_clean'],
      dtype='object')

In [5]:
goodreads.columns

Index(['id', 'title', 'author', 'rating', 'numratings', 'language', 'genres',
       'bookformat', 'edition', 'page_count', 'publisher', 'publish_year',
       'price', 'isbn_clean'],
      dtype='object')

In [ ]:
from rapidfuzz import fuzz
from sentence_transformers import SentenceTransformer, util


# embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

def similarity_score(src, tgt):
    """Compute semantic + fuzzy similarity between column names."""
    emb_src = model.encode(src, convert_to_tensor=True)
    emb_tgt = model.encode(tgt, convert_to_tensor=True)
    semantic_sim = util.cos_sim(emb_src, emb_tgt).item()
    fuzzy_sim = fuzz.token_sort_ratio(src, tgt) / 100.0
    return 0.6 * semantic_sim + 0.4 * fuzzy_sim


def schema_match_semantic(source_cols, target_cols, threshold=0.6):
    """Find best target column for each source column."""
    mapping = {}
    for src in source_cols:
        best_tgt, best_score = None, 0
        for tgt in target_cols:
            score = similarity_score(src, tgt)
            if score > best_score:
                best_score, best_tgt = score, tgt
        mapping[src] = best_tgt if best_score >= threshold else None
    return mapping

/Users/abd/Developer/team_project_data_integration/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
def unify_schemas(schema_dict, target_schema_name="metabooks"):
    """
    Use one dataset (e.g., 'metabooks') as target schema.
    Match all other schemas to it.
    """
    if target_schema_name not in schema_dict:
        raise ValueError(f"Target schema '{target_schema_name}' not found in schema_dict")

    target_cols = schema_dict[target_schema_name]
    mappings = {}

    for name, cols in schema_dict.items():
        if name == target_schema_name:
            # Target schema maps to itself
            mappings[name] = {col: col for col in cols}
        else:
            mappings[name] = schema_match_semantic(cols, target_cols)

    return target_cols, mappings

In [ ]:
schemas = {
    "amazon": amazon_books.columns,
    "metabooks": metabooks.columns,
    "goodreads": goodreads.columns
}

target_schema, mappings = unify_schemas(schemas, target_schema_name="goodreads")

print("Target Schema:")
print(target_schema)

print("\nSchema Mappings:")
for name, mapping in mappings.items():
    print(f"\n{name}:")
    for k, v in mapping.items():
        print(f"  {k:20} -> {v}")

Target Schema (metabooks):
Index(['id', 'title', 'author', 'rating', 'numratings', 'language', 'genres',
       'bookformat', 'edition', 'page_count', 'publisher', 'publish_year',
       'price', 'isbn_clean'],
      dtype='object')

Schema Mappings:

amazon:
  id                   -> id
  title                -> title
  book-author          -> author
  year-of-publication  -> publish_year
  publisher            -> publisher
  isbn_clean           -> isbn_clean

metabooks:
  id                   -> id
  title                -> title
  author_name          -> author
  average_rating       -> rating
  language             -> language
  genres               -> genres
  publisher            -> publisher
  page_count           -> page_count
  price                -> price
  publish_year         -> publish_year
  isbn_clean           -> isbn_clean

goodreads:
  id                   -> id
  title                -> title
  author               -> author
  rating               -> rating
  numra

In [9]:
def apply_schema_mapping(df, mapping):
    rename_dict = {k: v for k, v in mapping.items() if v is not None}
    return df.rename(columns=rename_dict)

amazon_new = apply_schema_mapping(amazon_books, mappings["amazon"])
metabooks_new = apply_schema_mapping(metabooks, mappings["metabooks"])
goodreads_new = apply_schema_mapping(goodreads, mappings["goodreads"])

In [13]:
amazon_new.to_parquet(DATA_DIR / "amazon_new.parquet", index=False, compression="snappy")
metabooks_new.to_parquet(DATA_DIR / "goodreads_new.parquet", index=False, compression="snappy")
goodreads_new.to_parquet(DATA_DIR / "metabooks_new.parquet", index=False, compression="snappy")